# Looping Graph

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Dict, List
import random

In [2]:
class AgentState(TypedDict):
    name: str
    number: List[int]
    counter: int

In [3]:
def greeting_node(state: AgentState) -> AgentState:
    """Greeting Node which says hi to the person"""
    
    state["name"] = f"Hi there, {state['name']}!"
    state['counter'] = 0
    
    return state


def random_node(state: AgentState) -> AgentState:
    """Generate a random number from 0 to 10"""
    
    state["number"].append(random.randint(0, 10))
    state['counter'] += 1
    
    return state


def should_continue(state: AgentState) -> bool:
    """Decide whether to continue generating random numbers or not"""
    
    if state['counter'] < 5:
        print("ENTERING LOOP", state["counter"])
        return "loop" # Continue looping
    else:
        return "exit" # Exit the loop
    



In [5]:
# greeting -> random -> random -> random -> random -> random -> "END"

graph = StateGraph(AgentState)

graph.add_node("greeting", greeting_node)
graph.add_node("random", random_node)

graph.add_edge(START, "greeting")
graph.add_edge("greeting", "random")
# graph.add_conditional_edges(
#     "random", # Source node
#     should_continue, # Routing function
#     {
#         "loop": "random", # self loop back to same node
#         "exit": END # end the graph
#     }
# )

graph.add_conditional_edges(
    "random", # Source node
    lambda state: "loop" if state["counter"] < 5 else "exit", # Routing function
    {
        "loop": "random", # self loop back to same node
        "exit": END # end the graph
    }
)

app = graph.compile()

In [9]:
result = app.invoke({"name": "Reda", "number": [], "counter": 0})
print(result)

{'name': 'Hi there, Reda!', 'number': [5, 9, 6, 9, 2], 'counter': 5}
